In [1]:
using Oscar

# Define Rings/Compute Gr\"obner bases

In [22]:
# use cyclotomic_field(4) to include i = z_4 = sqrt(-1) in QQ
# slows down the code and might be nice to avoid but is useful for testing the recurrence relation
CC,z_4 = cyclotomic_field(4)

(Cyclotomic field of order 4, z_4)

In [23]:
z_4*z_4

-1

In [24]:
# Define the coefficient field and its variables
T, (mu0, mu1, mu2, E) = polynomial_ring(CC, [:mu0, :mu1, :mu2, :E])
K = fraction_field(T)

# define graded polynomial ring and weighted homogeneous polynomial f
R, (x, y, z) = graded_polynomial_ring(K, [:x, :y, :z]; weights = [1,2,1])

#  treat mu1 and mu2 as order hbar quantum corrections
# f = K(1//2)*y^2 + x^4 + (E - K(2))*x^2*z^2  + (mu0 - E + K(1))*z^4
f = K(1//2)*y^2 - x^4 + (E + K(1))*x^2*z^2  + (mu0 - E )*z^4



# the quantum corrections
# for convenience consider mu2 -> i mu2 and mu1 -> i mu1 
# this ensures the trick used to account for the i appearing in the recurrence for the quantum periods still applies
# note: analytic continuation back to real values should be trivial
v1 = -mu2*x^3*z + (mu1 + mu2)*x*z^3 


# compute ideal and gb plus a transformation matrix (i believe default ordering is degrevlex)
J = ideal(R, [derivative(f,xs) for xs in gens(R)])
gb,C = groebner_basis_with_transformation_matrix(J)



# define t_0
S_0 , (x0, y0, z0) = graded_polynomial_ring(K, [:x0, :y0, :z0],weights = [1,1,1])
t_0 = hom(R,S_0,[x0,y0*(z0-x0),z0])

#NOTE J_0 ≠ t_0(J)
J_0 = ideal(S_0, [derivative(t_0(f),xs) for xs in gens(S_0)])
gb_0,C_0 = groebner_basis_with_transformation_matrix(J_0)

# coordinates of p in the monomial basis
function coords_in_monomial_basis(p, basis::Vector)
    return [coeff(p, m) for m in basis]
end

coords_in_monomial_basis (generic function with 1 method)

# Griffiths-Dwork Reduction

In [25]:
# rescales a homogeneous polynomial by its pole order defined by the polynomial f and the graded ring R
# extends, by linearity, to all polynomials "poly"
function mhat(poly,f)
    R = parent(f)
    w = sum(weights(R))[1]
    degf = total_degree(f)
    hcs = homogeneous_components(R(poly))
    return sum( hcs[k]*(k[1]+w)/degf for k in keys(hcs);init=0 )
end

# rescales a homogeneous polynomial by the reciprocal of its pole order.
# extends, by linearity, to all polynomials "poly"
function mhat_Inverse(poly,f)
    R = parent(f)
    w = sum(weights(R))[1]
    degf = total_degree(f)
    hcs = homogeneous_components(R(poly))
    return sum( hcs[k]/(k[1]+w)*degf for k in keys(hcs);init=0 )
end

# implements a single step in the reduction of pole order 
# the inputs are given as follows:
# p is a polynomial in R, a graded_polynomial_ring over K, which represents the numerator of a differential with pole locus f=0
# gb is a gr{\"o}bner basis for the jacobian ideal of factor
# C is a transformation matrix which converts quotients found with gb to the desired quotients 
function PoleReduce(p , f, gb, C)
    R = parent(f)
    xs = gens(R)
    q,r = reduce_with_quotients(p,gb)
    q = C*transpose(q)
    q = mhat_Inverse(sum([derivative(q[i],xs[i]) for i in 1:length(xs)] ),f)
    return q+r
end

# calculates the total_degree of a polynomial accounting for grading
weighted_degree(p) = maximum([k[1] for k in keys(homogeneous_components(p))])

# applies PoleReduce until the polynomial is completely reduced
function PoleReduceFull(poly,f,gb,C)
    R = parent(f)
    w = sum(weights(R))[1]
    d = (weighted_degree(R(poly)) +w)/total_degree(f)
    res = poly
    for _ in 1:d
        new_res = PoleReduce(res, f, gb, C)
        if new_res == res
            break
        end
        res = new_res
    end
    return res
end


# produces a monomial basis for R/J and selects the terms with proper grading so as to represent differential forms
function rational_de_Rham_basis(f,R)
    J = ideal(R, [derivative(f,xs) for xs in gens(R)])
    A,p = quo(R,J)
    bas = monomial_basis(A)
    w = sum(weights(R))[1]
    i = 1
    out = Vector{typeof(bas[1])}()
    for b in bas 
        if isinteger((degree(b)[1]+w)//degree(f)[1])
            push!(out,b)
        end
    end
    return out
end


param_diff(p,u) = sum(derivative(r[1],u)*prod(gens(R).^r[2]) for r in collect(coefficients_and_exponents(p)))

# defines the (iterated application of derivatives to the polynomials)
# p is a polynomial in R, a graded_polynomial_ring over K, which represents the numerator of a differential with pole locus f=0
# us = [u1,u2,...] with ui in gens(K)
# computes the derivative \partial^n/\partial u_1 ... \partial u_n 
function ∇_GM(p,us,f)
    R = parent(f)
    param_diff(p,u) = sum(derivative(r[1],u)*prod(gens(R).^r[2]) for r in collect(coefficients_and_exponents(p)))
    ∇(p,u) = param_diff(R(p),u) - param_diff(f,u)*mhat(p,f)
    pnew = p 
    for u in us
        pnew = ∇(pnew,u)
    end
    return pnew
end


# convenient functions for the analysis below
    
# the function g_y(n,m) provides numerical factor used in y_reduce below
function g_y(n,m)
    if n==0 
        out = 1
    elseif n % 2 == 0
        out= prod((n-2*i-1)//(m-i-1) for i in 0:(n//2-1))
    else
        out=0
    end
    return out
end


# computes the reduction of a polynomial y^n * p(x,z) using the reduction of pole order to eliminate all powers of y
function y_reduce(poly,f)
    R = parent(f)
    if poly == 0 
        return 0
    else 
    w=weights(R)
    pole_ord(ints) = (sum(w[i][1]*ints[i] for i in 1:length(w)) + sum(w)[1])//total_degree(f) 
    exps = collect(exponents(poly))
    cs = collect(coefficients(poly))
    return sum(cs[i]*g_y(exps[i][2], pole_ord(exps[i]) )*x^exps[i][1]*z^exps[i][3] for i in 1:length(cs) )
    end
end    


t_0_star(p) = PoleReduceFull((x0-z0)*t_0(p),t_0(f),gb_0,C_0)
M_1 = stack([coords_in_monomial_basis(t_0_star(b),[z0^5,z0,x0]) for b in [z^4,1]])

3×2 Matrix{AbstractAlgebra.Generic.FracFieldElem{AbstractAlgebra.Generic.MPoly{AbsSimpleNumFieldElem}}}:
 (-1//2*mu0*E - 1//2*mu0)//(mu0 + 1//2*E^2 - 1//2*E)  0
 (1//4*E)//(mu0 + 1//2*E^2 - 1//2*E)                  -1
 1//4//(mu0 + 1//2*E^2 - 1//2*E)                      1

# Define new basis given by the span of $N, \frac{\partial N}{\partial E},\frac{\partial N}{\partial \mu_0} $

## code

In [6]:
# coordinates of p in the monomial basis
function coords_in_monomial_basis(p, basis::Vector)
    return [coeff(p, m) for m in basis]
end

# compute change of basis
function change_of_basis(M,basis_1::Vector, basis_2::Vector)
    return inv(M(stack([coords_in_monomial_basis(b,basis_1) for b in basis_2] )))
end


function reduce_to_derivative_basis_R(poly,Mat)
    if poly==0
        return [K(0),K(0),K(0)]
    end
    red = PoleReduceFull(poly,f,gb,C)
    vec = coords_in_monomial_basis(R(red),[z^4,R(1)])
    return Mat*(M_1*vec)
end

function reduce_to_derivative_basis_S_0(poly,Mat)
    if poly==0
        return [K(0),K(0),K(0)]
    end
    red = PoleReduceFull(poly,t_0(f),gb_0,C_0)
    vec = coords_in_monomial_basis(red,[z0^5,z0,x0])
    return Mat*(vec)
end

# 4. Create a matrix space over the field K
M = matrix_space(K, 3, 3)


N_poly_form = [(-16*mu0^2 + 4*mu0*E^2 + 16*mu0*E - 16*mu0 - 4*E^3 + 4*E^2)//(E - 2)*z^4 +  (4*mu0 - 4*E + 4)//(E - 2) - 2*(E-1),
-2*mu0*z0
]

N = t_0_star(N_poly_form[1]) + N_poly_form[2]
dN_dE = PoleReduceFull(∇_GM( N,[E], t_0(f)),t_0(f),gb_0,C_0)
dN_dmu0 = PoleReduceFull(∇_GM( N,[mu0], t_0(f)),t_0(f),gb_0,C_0)

M_N_basis = change_of_basis(M,[z0^5,z0,x0],[N, dN_dE, dN_dmu0])

[                          (-1//8*mu0 + 1//16*E^2 - 1//16*E)//(mu0^3 - 1//4*mu0^2*E^2 - mu0^2*E + mu0^2 + 1//4*mu0*E^3 - 1//4*mu0*E^2)    0    0]
[              (3//8*mu0*E - 1//4*mu0 - 1//8*E^3 + 1//8*E^2)//(mu0^3 - 1//4*mu0^2*E^2 - mu0^2*E + mu0^2 + 1//4*mu0*E^3 - 1//4*mu0*E^2)    0   -1]
[(-1//4*mu0^2 + 5//8*mu0*E - 1//2*mu0 - 1//8*E^3 + 1//8*E^2)//(mu0^3 - 1//4*mu0^2*E^2 - mu0^2*E + mu0^2 + 1//4*mu0*E^3 - 1//4*mu0*E^2)   -1   -1]

In [7]:
M_N_basis*(M_1*coords_in_monomial_basis(R(1),[z^4,R(1)]))

3-element Vector{AbstractAlgebra.Generic.FracFieldElem{AbstractAlgebra.Generic.MPoly{AbsSimpleNumFieldElem}}}:
 0
 -1
 0

In [8]:
M_1

3×2 Matrix{AbstractAlgebra.Generic.FracFieldElem{AbstractAlgebra.Generic.MPoly{AbsSimpleNumFieldElem}}}:
 (1//2*mu0*E - mu0)//(mu0 - 1//2*E^2 + 1//2*E)  0
 (-1//4*E + 1//4)//(mu0 - 1//2*E^2 + 1//2*E)    -1
 1//4//(mu0 - 1//2*E^2 + 1//2*E)                1

## Picard-Fuchs equations:

the vector $\vec v$ below is such that 
$$
    \frac{\partial^2 N}{\partial E^2} = v_1 N + v_2 \frac{\partial N}{\partial E} + v_3 \frac{\partial N}{\partial \mu_0}
$$

In [9]:
reduce_to_derivative_basis_S_0(∇_GM( N,[E,E], t_0(f)),M_N_basis)

3-element Vector{AbstractAlgebra.Generic.FracFieldElem{AbstractAlgebra.Generic.MPoly{AbsSimpleNumFieldElem}}}:
 (1//8*mu0 - 1//16*E)//(mu0^2 - 1//4*mu0*E^2 - mu0*E + mu0 + 1//4*E^3 - 1//4*E^2)
 (-1//8*mu0*E + 1//4*mu0)//(mu0^2 - 1//4*mu0*E^2 - mu0*E + mu0 + 1//4*E^3 - 1//4*E^2)
 (-1//4*mu0^2 + 1//8*mu0*E)//(mu0^2 - 1//4*mu0*E^2 - mu0*E + mu0 + 1//4*E^3 - 1//4*E^2)

Check

In [10]:
reduce_to_derivative_basis_R(∇_GM( -1,[E], f),M_N_basis)-reduce_to_derivative_basis_S_0(∇_GM( N,[E,E], t_0(f)),M_N_basis)

3-element Vector{AbstractAlgebra.Generic.FracFieldElem{AbstractAlgebra.Generic.MPoly{AbsSimpleNumFieldElem}}}:
 0
 0
 0

the vector $\vec v$ below is such that 
$$
    \frac{\partial^2 N}{\partial E \partial \mu_0} = v_1 N + v_2 \frac{\partial N}{\partial E} + v_3 \frac{\partial N}{\partial \mu_0}
$$

In [11]:
reduce_to_derivative_basis_S_0(∇_GM( N,[E,mu0], t_0(f)),M_N_basis)

3-element Vector{AbstractAlgebra.Generic.FracFieldElem{AbstractAlgebra.Generic.MPoly{AbsSimpleNumFieldElem}}}:
 (-1//16*E + 1//8)//(mu0^2 - 1//4*mu0*E^2 - mu0*E + mu0 + 1//4*E^3 - 1//4*E^2)
 (-1//4*mu0 + 1//8*E^2 - 1//8*E)//(mu0^2 - 1//4*mu0*E^2 - mu0*E + mu0 + 1//4*E^3 - 1//4*E^2)
 (1//8*mu0*E - 1//4*mu0)//(mu0^2 - 1//4*mu0*E^2 - mu0*E + mu0 + 1//4*E^3 - 1//4*E^2)

the vector $\vec v$ below is such that 
$$
    \frac{\partial^2 N}{\partial \mu_0^2} = v_1 N + v_2 \frac{\partial N}{\partial E} + v_3 \frac{\partial N}{\partial \mu_0}
$$

In [12]:
reduce_to_derivative_basis_S_0(∇_GM( N,[mu0,mu0], t_0(f)),M_N_basis)

3-element Vector{AbstractAlgebra.Generic.FracFieldElem{AbstractAlgebra.Generic.MPoly{AbsSimpleNumFieldElem}}}:
 (-1//8*mu0 + 1//16*E^2 - 1//16*E)//(mu0^3 - 1//4*mu0^2*E^2 - mu0^2*E + mu0^2 + 1//4*mu0*E^3 - 1//4*mu0*E^2)
 (3//8*mu0*E - 1//4*mu0 - 1//8*E^3 + 1//8*E^2)//(mu0^3 - 1//4*mu0^2*E^2 - mu0^2*E + mu0^2 + 1//4*mu0*E^3 - 1//4*mu0*E^2)
 (-1//4*mu0^2 + 5//8*mu0*E - 1//2*mu0 - 1//8*E^3 + 1//8*E^2)//(mu0^3 - 1//4*mu0^2*E^2 - mu0^2*E + mu0^2 + 1//4*mu0*E^3 - 1//4*mu0*E^2)

# Compute Periods

In [27]:
PoleReduceFull(x^2*z^2-z^4,f,gb,C)

(-2*mu0 + E - 1)//(E + 1)*z^4 + 1//2//(E + 1)

In [28]:
function mhat_minus_one_Inverse(poly,f)
    R = parent(f)
    w = sum(weights(R))[1]
    degf = total_degree(f)
    hcs = homogeneous_components(R(poly))
    return sum( hcs[k]/(k[1]+w-degf)*degf  for k in keys(hcs);init=0 )
end

# caution broken for most arguments
function int_dE(poly,f)
    R = parent(f)
    dfdE = x^2*z^2 - z^4
    q,r = reduce_with_quotients(-mhat_minus_one_Inverse(poly,f),[dfdE])
    q_new = [q[1],y_reduce(r,f)]
    q2,r = reduce_with_quotients(R(q_new[2]),[x*z^2])
    return reduce_to_derivative_basis_R(R(q_new[1]),M_N_basis) + reduce_to_derivative_basis_S_0(t_0(q2[1]),M_N_basis)
end

int_dE (generic function with 1 method)

In [29]:
# checking definitions g_2_old works new def does not
delta1(U) = z * (derivative(U, x) - derivative(f, x) * mhat(U,f))
delta2(U) = (derivative(U, y) - derivative(f, y) * mhat(U,f))

 g_1_old(U) = z_4*y*((z^2 - x^2) * delta1(U) -
                       z*x * y * delta2(U)) 
half = K(1//2)
 g_2_old(U) = (half*z^4 - z^2*x^2 + half*x^4) * delta1(delta1(U)) +
            y*z*( x^3 - z^2*x ) * delta2(delta1(U)) +
            half*y^2*x^2 * delta2(z^2 * delta2(U)) +
            half*z*(-z^2 + x^2) * (y*z*delta2(U) + x*delta1(U))

        
δ_x(U) = z * (derivative(U, x) - derivative(f, x) * mhat(U,f))
δ_y(U) = y * (derivative(U, y) - y * mhat(U,f))
δ_q(U) = (z^2 - x^2)*δ_x(U) - x*z*δ_y(U)

g_1(U) = I*y*δ_q(U)
g_2(U) = (1//2)*(δ_q(δ_q(U)) + (1//2)*x*z*δ_q(U))


g_2 (generic function with 1 method)

In [32]:
g_2_old(R(1))-mu0*g_2(R(1))

(-20*mu0 + 16)*x^10*z^2 + (20*mu0*E + 60*mu0 - 16*E - 48)*x^8*z^4 + (10*mu0 - 8)*x^6*y^2*z^2 + (-5*mu0*E^2 - 50*mu0*E - 65*mu0 + 4*E^2 + 40*E + 52)*x^6*z^6 + (-9*mu0 + 8)*x^6*z^2 + (-5*mu0*E - 15*mu0 + 4*E + 12)*x^4*y^2*z^4 + (10*mu0*E^2 + 40*mu0*E + 30*mu0 - 8*E^2 - 32*E - 24)*x^4*z^8 + (5//2*mu0*E + 35//2*mu0 - 2*E - 16)*x^4*z^4 + (-5//4*mu0 + 1)*x^2*y^4*z^2 + (5*mu0*E + 5*mu0 - 4*E - 4)*x^2*y^2*z^6 + (5//4*mu0 - 1)*x^2*y^2*z^2 + (-5*mu0*E^2 - 10*mu0*E - 5*mu0 + 4*E^2 + 8*E + 4)*x^2*z^10 + (-7//2*mu0*E - 19//2*mu0 + 3*E + 9)*x^2*z^6 + (-1//2*mu0 + 1//2)*y^2*z^4 + (mu0*E + mu0 - E - 1)*z^8

In [33]:
# compute quantum periods using above data.
# note that output is a vector of length max_iter which consists of vectors of the form [p_1,p_2] 
# where the p_1 is the integral of dPi/dE w.r.t. E and p_2 is the piece with no simple integral 

function Pi_gen(max_iter,f)

    R = parent(f)
    J = ideal(R, [derivative(f,xs) for xs in gens(R)])
    gb,C = groebner_basis_with_transformation_matrix(J)

    quarter = K(1//4)
    half    = K(1//2)
    v1 = -mu2*x^3*z + (mu1 + mu2)*x*z^3 
    # v1 = 0

    delta1(U) = z * (derivative(U, x) - derivative(f, x) * mhat(U,f))
    delta2(U) = (derivative(U, y) - derivative(f, y) * mhat(U,f))

    g_1(U) = z_4*y*((z^2 - x^2) * delta1(U) -
                       z*x * y * delta2(U)) 

    g_2(U) = (half*z^4 - z^2*x^2 + half*x^4) * delta1(delta1(U)) +
            y*z*( x^3 - z^2*x ) * delta2(delta1(U)) +
            half*y^2*x^2 * delta2(z^2 * delta2(U)) +
            half*z*(-z^2 + x^2) * (y*z*delta2(U) + x*delta1(U))


    # b_size = 1+max_iter
    a2 = R(1)
    a1 = R(0)
    
    # b = Vector{typeof(M_N_basis*coords_in_monomial_basis(N,[z0^5,z0,x0]))}(undef, b_size)
    # b[1] = M_N_basis*coords_in_monomial_basis(N,[z0^5,z0,x0])
    
    prev_prev = a1
    prev = a2
    for i in 3:2:(max_iter+2)
        # i is odd: compute a[i]
        curr_odd = g_1(prev) -v1*prev + g_2(prev_prev)
        println( curr_odd ) 
        println( PoleReduceFull(curr_odd,f,gb,C) ) 
        # println( y_reduce(curr_odd,f) )
        # b[i-1] = int_dE(curr_odd,f)
        # i+1 is even: compute a[i+1] and store in b
        if i + 1 <= max_iter+2
            curr_even = g_1(curr_odd) -v1*prev + g_2(prev)
            println(curr_even)
            println( PoleReduceFull(curr_even,f,gb,C) ) 
        
            # b[i]  = int_dE(curr_even,f)
            
            # Update state for next iteration
            prev_prev = curr_odd
            prev = curr_even
        end
    end
    # return b 
end

Pi_gen (generic function with 1 method)

In [34]:
Pi = Pi_gen(4,f)

-4*z_4*x^5*y*z + (2*z_4*E + 6*z_4)*x^3*y*z^3 + mu2*x^3*z + z_4*x*y^3*z + (-2*z_4*E - 2*z_4)*x*y*z^5 + (-mu1 - mu2)*x*z^3
0
-48*x^10*y^2*z^2 + 16*x^10*z^2 + (48*E + 144)*x^8*y^2*z^4 - 8*z_4*mu2*x^8*y*z^2 + (-16*E - 48)*x^8*z^4 + 24*x^6*y^4*z^2 + (-12*E^2 - 120*E - 156)*x^6*y^2*z^6 - 32*x^6*y^2*z^2 + (8*z_4*mu1 + 4*z_4*mu2*E + 20*z_4*mu2)*x^6*y*z^4 + (4*E^2 + 40*E + 52)*x^6*z^6 + 8*x^6*z^2 + (-12*E - 36)*x^4*y^4*z^4 + 2*z_4*mu2*x^4*y^3*z^2 + (24*E^2 + 96*E + 72)*x^4*y^2*z^8 + (12*E + 56)*x^4*y^2*z^4 + (-4*z_4*mu1*E - 12*z_4*mu1 - 8*z_4*mu2*E - 16*z_4*mu2)*x^4*y*z^6 - 3*z_4*mu2*x^4*y*z^2 + (-8*E^2 - 32*E - 24)*x^4*z^8 + (-2*E - 16)*x^4*z^4 + mu2*x^3*z - 3*x^2*y^6*z^2 + (12*E + 12)*x^2*y^4*z^6 + 5*x^2*y^4*z^2 + (-2*z_4*mu1 - 2*z_4*mu2)*x^2*y^3*z^4 + (-12*E^2 - 24*E - 12)*x^2*y^2*z^10 + (-14*E - 26)*x^2*y^2*z^6 - x^2*y^2*z^2 + (4*z_4*mu1*E + 4*z_4*mu1 + 4*z_4*mu2*E + 4*z_4*mu2)*x^2*y*z^8 + (z_4*mu1 + 4*z_4*mu2)*x^2*y*z^4 + (4*E^2 + 8*E + 4)*x^2*z^10 + (3*E + 9)*x^2*z^6 + (-mu1 - mu2)*x*z^3 

In [36]:
test= -48*x^10*y^2*z^2 + 16*x^10*z^2 + (-48*E + 192)*x^8*y^2*z^4 + 8*z_4*mu2*x^8*y*z^2 + (16*E - 64)*x^8*z^4 - 24*x^6*y^4*z^2 + (-12*E^2 + 144*E - 288)*x^6*y^2*z^6 + 32*x^6*y^2*z^2 + (-8*z_4*mu1 + 4*z_4*mu2*E - 24*z_4*mu2)*x^6*y*z^4 + (4*E^2 - 48*E + 96)*x^6*z^6 - 8*x^6*z^2 + (-12*E + 48)*x^4*y^4*z^4 + 2*z_4*mu2*x^4*y^3*z^2 + (24*E^2 - 144*E + 192)*x^4*y^2*z^8 + (12*E - 68)*x^4*y^2*z^4 + (-4*z_4*mu1*E + 16*z_4*mu1 - 8*z_4*mu2*E + 24*z_4*mu2)*x^4*y*z^6 - 3*z_4*mu2*x^4*y*z^2 + (-8*E^2 + 48*E - 64)*x^4*z^8 + (-2*E + 18)*x^4*z^4 + mu2*x^3*z - 3*x^2*y^6*z^2 + (12*E - 24)*x^2*y^4*z^6 + 5*x^2*y^4*z^2 + (-2*z_4*mu1 - 2*z_4*mu2)*x^2*y^3*z^4 + (-12*E^2 + 48*E - 48)*x^2*y^2*z^10 + (-14*E + 40)*x^2*y^2*z^6 - x^2*y^2*z^2 + (4*z_4*mu1*E - 8*z_4*mu1 + 4*z_4*mu2*E - 8*z_4*mu2)*x^2*y*z^8 + (z_4*mu1 + 4*z_4*mu2)*x^2*y*z^4 + (4*E^2 - 16*E + 16)*x^2*z^10 + (3*E - 12)*x^2*z^6 + (-mu1 - mu2)*x*z^3 - y^4*z^4 + (2*E - 4)*y^2*z^8 + 1//2*y^2*z^4 + (-z_4*mu1 - z_4*mu2)*y*z^6 + (-E + 2)*z^8
reduce_with_quotients(test,[param_diff(f,E)])


([-48*x^8*y^2+16*x^8+(-48*E+144)*x^6*y^2*z^2+8*z_4*mu2*x^6*y+(16*E-48)*x^6*z^2-24*x^4*y^4+(-12*E^2+96*E-144)*x^4*y^2*z^4+32*x^4*y^2+(-8*z_4*mu1+4*z_4*mu2*E-16*z_4*mu2)*x^4*y*z^2+(4*E^2-32*E+48)*x^4*z^4-8*x^4+(-12*E+24)*x^2*y^4*z^2+2*z_4*mu2*x^2*y^3+(12*E^2-48*E+48)*x^2*y^2*z^6+(12*E-36)*x^2*y^2*z^2+(-4*z_4*mu1*E+8*z_4*mu1-4*z_4*mu2*E+8*z_4*mu2)*x^2*y*z^4-3*z_4*mu2*x^2*y+(-4*E^2+16*E-16)*x^2*z^6+(-2*E+10)*x^2*z^2-3*y^6+5*y^4-2*z_4*mu1*y^3*z^2+(-2*E+4)*y^2*z^4-y^2+(z_4*mu1+z_4*mu2)*y*z^2+(E-2)*z^4], mu2*x^3*z + (-mu1 - mu2)*x*z^3 - 3*y^6*z^4 + 4*y^4*z^4 - 2*z_4*mu1*y^3*z^6 - 1//2*y^2*z^4)

In [35]:
PoleReduceFull(∇_GM(R(1),[E],f),f,gb,C)

(2*mu0 - E + 1)//(E + 1)*z^4 - 1//2//(E + 1)

In [94]:
GC.gc()